# Hadamard: making superpositions
**Quantum Computing · Week 2 (Gates and the Bloch sphere) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w2d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

The Hadamard gate H: it makes superpositions, undoes them, and swaps the z and x views of a qubit.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

H = Operator.from_label("H").data
print("H =\n", np.round(H, 3))
print("H·H = I ?", np.allclose(H @ H, np.eye(2)))

def ket(sv, labels=("|0>", "|1>")):
    """A state with real amplitudes, in Dirac notation."""
    out = ""
    for a, l in zip(sv.data, labels):
        if abs(a) < 1e-9:
            continue
        v = a.real
        out += f"{v:.3f}{l}" if not out else f" {'-' if v < 0 else '+'} {abs(v):.3f}{l}"
    return out

s = 1 / np.sqrt(2)
for name, amps in {"|0>": [1, 0], "|1>": [0, 1], "|+>": [s, s], "|->": [s, -s]}.items():
    print(f"H {name:>3} = {ket(Statevector(amps).evolve(Operator(H)))}")

## H swaps the z axis and the x axis
On the Bloch sphere, H is a half-turn about the diagonal between x and z:
the arrow $(x, y, z)$ becomes $(z, -y, x)$.

In [ ]:
def bloch(sv):
    a, b = sv.data
    c = np.conj(a) * b
    return np.round([2 * c.real, 2 * c.imag, abs(a) ** 2 - abs(b) ** 2], 3)

psi = Statevector([np.cos(0.5), np.exp(1.1j) * np.sin(0.5)])
print("before H:", bloch(psi))
print("after H: ", bloch(psi.evolve(Operator(H))))

## One H is a coin, two H's are not

In [ ]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler(seed=1)
one, two = QuantumCircuit(1), QuantumCircuit(1)
one.h(0)
two.h(0)
two.h(0)
for qc in (one, two):
    qc.measure_all()
res = sampler.run([one, two], shots=1000).result()
print("H, measure:   ", res[0].data.meas.get_counts())
print("H, H, measure:", res[1].data.meas.get_counts())

Why? Follow the amplitudes of $HH|0\rangle$: the first H gives $\frac{1}{\sqrt2}(|0\rangle + |1\rangle)$.
The second H sends $|0\rangle \to \frac{1}{\sqrt2}(|0\rangle + |1\rangle)$ and $|1\rangle \to \frac{1}{\sqrt2}(|0\rangle - |1\rangle)$.
Adding up: $|0\rangle$ gets $\frac12 + \frac12 = 1$, and $|1\rangle$ gets $\frac12 - \frac12 = 0$. The two routes to 1 cancel.

## H on many qubits: every bitstring at once

In [ ]:
import matplotlib.pyplot as plt

qc3 = QuantumCircuit(3)
qc3.h([0, 1, 2])                     # one H on each qubit
print(np.round(Statevector(qc3).probabilities(), 3), "<- 8 outcomes, 1/8 each")
qc3.measure_all()
counts = sampler.run([qc3], shots=4000).result()[0].data.meas.get_counts()
keys = sorted(counts)
plt.figure(figsize=(6, 3))
plt.bar(keys, [counts[k] for k in keys], color="#ff8fb8")
plt.axhline(500, color="#7b5cfa", ls="--", label="4000 / 8 = 500")
plt.xlabel("bitstring (q2 q1 q0)")
plt.ylabel("counts")
plt.legend()
plt.show()

## Your turn: H changes the point of view
Check with `Operator` that $HZH = X$ and $HXH = Z$. Then predict $HYH$ before computing it.

In [ ]:
X, Y, Z = (Operator.from_label(p).data for p in "XYZ")
print("HZH = X ?", None)    # TODO: np.allclose(H @ Z @ H, X)
print("HXH = Z ?", None)    # TODO
HYH = None                  # TODO: compute it, then compare with Y and -Y
print("HYH =", HYH)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
print("HZH = X ?", np.allclose(H @ Z @ H, X))
print("HXH = Z ?", np.allclose(H @ X @ H, Z))
HYH = H @ Y @ H
print("HYH = -Y ?", np.allclose(HYH, -Y))
print("H swaps the x and z axes and flips y, so a turn about z, seen through H, is a turn about x.")